In [25]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool

In [2]:
from langchain.agents import create_react_agent,AgentExecutor

In [26]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


In [9]:
search_tool = TavilySearchResults(max_results=2)

In [27]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [10]:
search_tool.invoke("Give me the latest news on AI")

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'AI and Us, AI Business Strategy, AI in Action, AI Market Trends, Cybersecurity AI, Data Engineering & MLOps, Deep Dives, Featured News, Features, Governance, Regulation & Policy, How It Works, Inside AI, Natural Language Processing (NLP), Open-Source & Democratised AI, Trust, Bias & Fairness, World of Work\n\n### IBM launches AI platform Bob to regulate SDLC costs\n\n## Latest\n\nGoogle logo as the company is folding Display Ads into its AI-powered Demand Gen platform, marking the end of a long-standing digital advertising model.\n\nMarketing AI\n\nMay 27, 2026\n\n# Google folds Display Ads into AI-first Demand Gen platform\n\nAutonomous AI systems test governance in physical environments\n\nAI in Action\n\nMay 26, 2026\n\n# Autonomous AI systems test governance in physical environments [...] Resource\n\nDecember 11, 2025\n\n# On-Demand Webinar: Turning a Hacker’s Toolkit Against Them\n\nResource\n\nDecember 11, 2025

In [11]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [13]:
response=llm.invoke("Tell me a joke about ai?")
response

AIMessage(content='Why did the AI program go on a diet?\n\nBecause it wanted to lose some bytes.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 42, 'total_tokens': 61, 'completion_time': 0.037909363, 'completion_tokens_details': None, 'prompt_time': 0.023562516, 'prompt_tokens_details': None, 'queue_time': 0.356039546, 'total_time': 0.061471879}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'finish_reason': 'stop', 'logprobs': None}, id='run-7ae7cbf5-6be4-4091-ab7c-3a93a5e142bc-0', usage_metadata={'input_tokens': 42, 'output_tokens': 19, 'total_tokens': 61})

In [30]:
prompt = hub.pull("hwchase17/react")

c:\Users\srith\anaconda3\envs\langagent\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [31]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [34]:
tools = [search_tool,get_weather_data]


In [ ]:
agent = create_react_agent(
    llm=llm,
    tools=tools,    
    prompt=prompt
)

In [36]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [37]:
response = agent_executor.invoke({
    "input": (
        "what is the current weather in vizag"
    )
})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: To find the current weather in Vizag, I need to use a function that can provide me with real-time weather information for a specific city. 

Action: get_weather_data
Action Input: VizagCity: Vizag
Temperature: 29°C
Weather: Partly Cloudy 
Humidity: 64%Thought: I have obtained the current weather information for Vizag, which includes the temperature, weather conditions, and humidity. This is all the information I need to answer the question.

Final Answer: The current weather in Vizag is Partly Cloudy with a temperature of 29°C and humidity of 64%.

> Finished chain.


In [38]:
print(response["output"])

The current weather in Vizag is Partly Cloudy with a temperature of 29°C and humidity of 64%.
